In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from sklearn.preprocessing import StandardScaler
from utils import train, validate

In [3]:
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Is CUDA available? True
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
train_df = pd.read_csv('data/train_dataset.csv')
val_df = pd.read_csv('data/validation_dataset.csv')

x_train = train_df.drop('0_y', axis=1).values
y_train = train_df['0_y'].values
x_val = val_df.drop('0_y', axis=1).values
y_val = val_df['0_y'].values

print(y_val)

[0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1]


In [5]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

In [6]:
x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

x_val_t = torch.tensor(x_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

print(y_train_t.min(), y_train_t.max())
print(y_val_t.min(), y_val_t.max())

tensor(0.) tensor(1.)
tensor(0.) tensor(1.)


In [7]:
BATCH_SIZE = 32
train_ds = TensorDataset(x_train_t, y_train_t)
val_ds = TensorDataset(x_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [8]:
model = nn.Sequential(
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, 1)
)
model = model.to(device)
model

Sequential(
  (0): Linear(in_features=1024, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=128, bias=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=1, bias=True)
)

In [9]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [10]:
epochs = 8
for epoch in range(epochs):
    print(f"Epoch {epoch}")
    train(model, loss_function, optimizer, train_loader, device)
    validate(model, loss_function, val_loader, device)

Epoch 0
Train - Loss: 17.1640 Accuracy: 0.9576
Valid - Loss: 0.0337 Accuracy: 1.0000
Epoch 1
Train - Loss: 7.5015 Accuracy: 0.9835
Valid - Loss: 0.0324 Accuracy: 1.0000
Epoch 2
Train - Loss: 3.7297 Accuracy: 0.9914
Valid - Loss: 0.0058 Accuracy: 1.0000
Epoch 3
Train - Loss: 2.1916 Accuracy: 0.9950
Valid - Loss: 0.1709 Accuracy: 0.9375
Epoch 4
Train - Loss: 4.3137 Accuracy: 0.9900
Valid - Loss: 0.0907 Accuracy: 0.9375
Epoch 5
Train - Loss: 1.5835 Accuracy: 0.9971
Valid - Loss: 0.0768 Accuracy: 0.9375
Epoch 6
Train - Loss: 0.8856 Accuracy: 0.9979
Valid - Loss: 0.0401 Accuracy: 1.0000
Epoch 7
Train - Loss: 0.2439 Accuracy: 0.9994
Valid - Loss: 0.0033 Accuracy: 1.0000


In [11]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}